In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

class SSMCore(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model  = d_model
        self.d_inner  = d_model * expand
        self.d_state  = d_state
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                  padding=d_conv-1, groups=self.d_inner, bias=True)
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj  = nn.Linear(1, self.d_inner, bias=True)
        self.A_log    = nn.Parameter(torch.randn(self.d_inner, d_state))
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        B, L, _  = x.shape
        xz       = self.in_proj(x)
        x_in, z  = xz.chunk(2, dim=-1)
        x_in     = self.conv1d(x_in.transpose(1,2))[:,:,:L].transpose(1,2)
        x_in     = F.silu(x_in)
        x_dbl    = self.x_proj(x_in)
        dt, B_p, C = x_dbl.split([1, self.d_state, self.d_state], dim=-1)
        dt       = F.softplus(self.dt_proj(dt))
        A        = -torch.exp(self.A_log)
        h        = torch.zeros(B, self.d_inner, self.d_state,
                               device=x.device, dtype=x_in.dtype)
        out      = torch.zeros(B, L, self.d_inner,
                               device=x.device, dtype=x_in.dtype)
        for t in range(L):
            dA_t      = torch.exp(dt[:,t,:].unsqueeze(-1) * A)
            dB_t      = dt[:,t,:].unsqueeze(-1) * B_p[:,t,:].unsqueeze(1)
            h         = h * dA_t + dB_t * x_in[:,t,:].unsqueeze(-1)
            out[:,t]  = (h * C[:,t,:].unsqueeze(1)).sum(-1)
        out = out + x_in * self.D
        out = out * F.silu(z)
        return self.out_proj(out)


class FourDirScan(nn.Module):
    def forward(self, x):
        x  = rearrange(x, 'b c h w -> b h w c')
        B, H, W, C = x.shape
        s1 = x.reshape(B, H*W, C)
        s2 = s1.flip(1)
        s3 = x.permute(0,2,1,3).reshape(B, H*W, C)
        s4 = s3.flip(1)
        return [s1, s2, s3, s4], H, W


class VMambaBlock(nn.Module):
    def __init__(self, dim, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm  = nn.LayerNorm(dim)
        self.scan  = FourDirScan()
        self.ssms  = nn.ModuleList([
            SSMCore(dim, d_state, d_conv, expand) for _ in range(4)
        ])
        self.merge = nn.Linear(dim * 4, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn   = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        shortcut   = x
        x_norm     = rearrange(x, 'b c h w -> b h w c')
        x_norm     = self.norm(x_norm)
        x_norm     = rearrange(x_norm, 'b h w c -> b c h w')
        scans, H, W = self.scan(x_norm)
        outs       = [ssm(scans[i]) for i, ssm in enumerate(self.ssms)]
        merged     = torch.cat(outs, dim=-1)
        merged     = self.merge(merged)
        merged     = rearrange(merged, 'b (h w) c -> b c h w', h=H, w=W)
        x          = shortcut + merged
        x_ffn      = rearrange(x, 'b c h w -> b h w c')
        x_ffn      = self.norm2(x_ffn)
        x_ffn      = self.ffn(x_ffn)
        x_ffn      = rearrange(x_ffn, 'b h w c -> b c h w')
        return x + x_ffn


class VMambaSmallEncoder(nn.Module):
    def __init__(self, patch_size=4, in_chans=3):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, 96, kernel_size=patch_size, stride=patch_size),
            nn.BatchNorm2d(96)
        )
        self.stage1 = nn.Sequential(*[VMambaBlock(96)  for _ in range(2)])
        self.down1  = nn.Conv2d(96,  192, kernel_size=2, stride=2)
        self.stage2 = nn.Sequential(*[VMambaBlock(192) for _ in range(2)])
        self.down2  = nn.Conv2d(192, 384, kernel_size=2, stride=2)
        self.stage3 = nn.Sequential(*[VMambaBlock(384) for _ in range(4)])
        self.down3  = nn.Conv2d(384, 768, kernel_size=2, stride=2)
        self.stage4 = nn.Sequential(*[VMambaBlock(768) for _ in range(2)])
        self.norm   = nn.LayerNorm(768)

    def forward(self, x):
        x = self.patch_embed(x)   # [B, 96,  64, 64]
        x = self.stage1(x)
        x = self.down1(x)         # [B, 192, 32, 32]
        x = self.stage2(x)
        x = self.down2(x)         # [B, 384, 16, 16]
        x = self.stage3(x)
        x = self.down3(x)         # [B, 768,  8,  8]
        x = self.stage4(x)
        x = rearrange(x, 'b c h w -> b h w c')
        x = self.norm(x)
        x = rearrange(x, 'b h w c -> b c h w')
        return x                  # [B, 768, 8, 8]


if __name__ == '__main__':
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    encoder = VMambaSmallEncoder().to(device)

    x   = torch.randn(2, 3, 256, 256).to(device)
    out = encoder(x)

    print(f"Input:  {x.shape}")
    print(f"Output: {out.shape}")
    print(f"Params: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M")
    print(f"Device: {out.device}")